In [1]:
"""
Modelo de Aprendizaje Profundo para Predicción de Interacciones miRNA-lncRNA
Versión con Parámetros Fijos y Análisis Exhaustivo + Exportación Excel de Salidas Softmax

Este script implementa un modelo basado en transformers y redes convolucionales para predecir
interacciones entre miRNA y lncRNA usando parámetros optimizados previamente.

Características principales:
- Arquitectura híbrida transformer-CNN con parámetros fijos optimizados
- Entrenamiento por 1000 épocas con early stopping
- Análisis exhaustivo con múltiples visualizaciones
- Evaluación completa de todos los datos con exportación a Excel
- Matrices de confusión detalladas
- Visualización de la arquitectura del modelo
- Exportación de todas las salidas del softmax en archivos Excel (.xlsx)

Autor: [Tu Nombre]
Fecha: [Fecha de Creación/Actualización]
"""

import os
import gc
import json
import time
import traceback
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import (roc_curve, auc, precision_recall_curve, average_precision_score, 
                            f1_score, confusion_matrix, accuracy_score, precision_score, recall_score,
                            classification_report, roc_auc_score)
import seaborn as sns
from tqdm import tqdm
import psutil
import warnings
warnings.filterwarnings('ignore')

# Intentar importar wandb; si no está disponible, crear un sustituto
try:
    import wandb
    WANDB_AVAILABLE = True
except ImportError:
    WANDB_AVAILABLE = False
    print("Weights & Biases (wandb) no está disponible. Se desactivará el registro de experimentos.")
    class WandbMock:
        def init(self, *args, **kwargs):
            return self
        def log(self, *args, **kwargs):
            pass
        def finish(self):
            pass
        def Image(self, *args, **kwargs):
            return None
        def Table(self, *args, **kwargs):
            return None
    wandb = WandbMock()

#------------------------------------------------------------------------------
# Configuración Global
#------------------------------------------------------------------------------

# Configuración de semilla para reproducibilidad
SEED = 42
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Configurar rutas
DATA_PATH = "../Data_Demo/embedding/"
RESULTS_PATH = "./outputs/"

# Crear directorios necesarios
os.makedirs(RESULTS_PATH, exist_ok=True)
os.makedirs(f"{RESULTS_PATH}plots", exist_ok=True)
os.makedirs(f"{RESULTS_PATH}analysis", exist_ok=True)
os.makedirs(f"{RESULTS_PATH}model_analysis", exist_ok=True)

# Configuración de estilo para visualizaciones
plt.style.use('default')
plt.rcParams['figure.figsize'] = (12, 8)
plt.rcParams['font.size'] = 12

# Parámetros optimizados del modelo
OPTIMIZED_PARAMS = {
    'batch_size': 2048,
    'cnn_activation': 'relu',
    'cnn_filters1': 128,
    'cnn_filters2': 128,
    'cnn_kernel_size': 5,
    'cnn_pool_size': 3,
    'dropout_rate': 0.5229,
    'embedding_dim': 64,
    'learning_rate': 0.001,
    'ff_dim_factor': 6,
    'num_heads': 4,
    'num_transformer_blocks': 2,
    'use_regularization': True,
    'weight_decay': 0.0006191,
    'epochs': 1000,
    'patience_early': 50,
    'patience_reduce_lr': 20
}

#------------------------------------------------------------------------------
# Funciones de Monitoreo de Memoria
#------------------------------------------------------------------------------

def get_memory_usage():
    """Obtiene el uso actual de memoria en MB."""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def log_memory_usage(label="Current"):
    """Registra el uso actual de memoria con una etiqueta."""
    print(f"🧠 {label} Memory Usage: {get_memory_usage():.2f} MB")

def cleanup_memory():
    """Realiza limpieza de memoria."""
    gc.collect()
    tf.keras.backend.clear_session()
    log_memory_usage("Después de limpieza")

#------------------------------------------------------------------------------
# Funciones de Carga y Procesamiento de Datos
#------------------------------------------------------------------------------

def cargar_datos():
    """
    Carga todos los archivos de datos y etiquetas como float32.
    """
    print("=== CARGANDO TODOS LOS DATOS ===")
    log_memory_usage("Antes de cargar datos")
    
    datos = {}
    
    # Lista de archivos .npy en el directorio
    archivos_npy = [f for f in os.listdir(DATA_PATH) if f.endswith('.npy')]
    print(f"Se encontraron {len(archivos_npy)} archivos .npy")
    
    # Identificar archivos que contienen objetos (primera columna con texto)
    archivos_object = [
        'binding_features_lnc.npy',
        'mir_features.npy',
        'binding_features_miRNA_precursor_individual.npy',
        'lnc_features.npy'
    ]
    
    # Cargar cada archivo .npy
    for archivo in tqdm(archivos_npy, desc="Cargando archivos"):
        nombre_clave = os.path.splitext(archivo)[0]
        ruta_completa = os.path.join(DATA_PATH, archivo)
        
        try:
            if archivo in archivos_object:
                print(f"Procesando archivo con objetos: {archivo}")
                raw_data = np.load(ruta_completa, allow_pickle=True)
                
                numeric_data = []
                for row in raw_data:
                    if isinstance(row, str):
                        elements = row.split(',')
                        if len(elements) > 1:
                            numeric_values = [float(val) for val in elements[1:]]
                            numeric_data.append(numeric_values)
                    elif hasattr(row, '__len__'):
                        try:
                            float(row[0])
                            numeric_data.append(row.astype(np.float32))
                        except (ValueError, TypeError):
                            numeric_values = [float(val) for val in row[1:]]
                            numeric_data.append(numeric_values)
                
                datos[nombre_clave] = np.array(numeric_data, dtype=np.float32)
                del raw_data
                del numeric_data
                gc.collect()
                
            else:
                datos[nombre_clave] = np.load(ruta_completa)
                if 'mask' in archivo or 'padded' in archivo or 'normalized' in archivo:
                    if datos[nombre_clave].dtype != np.uint8:
                        datos[nombre_clave] = datos[nombre_clave].astype(np.float32)
                else:
                    datos[nombre_clave] = datos[nombre_clave].astype(np.float32)
            
            print(f"  Cargado {archivo}: {datos[nombre_clave].shape}, {datos[nombre_clave].dtype}")
            
        except Exception as e:
            print(f"Error al cargar {archivo}: {e}")
        
        if len(datos) % 5 == 0:
            gc.collect()
    
    # Cargar etiquetas
    try:
        print("\nCargando etiquetas...")
        negative_pairs_path = os.path.join(DATA_PATH, "negative_pairs.txt")
        positive_pairs_path = os.path.join(DATA_PATH, "mirnas_lncrnas_validated_positive_pairs.txt")
        
        if not os.path.exists(positive_pairs_path):
            positive_pairs_path = "../Data_Demo/interactions/txt_interac_CNN/mirnas_lncrnas_validated_positive_pairs.txt"
        if not os.path.exists(negative_pairs_path):
            negative_pairs_path = "../Data_Demo/interactions/txt_interac_CNN/negative_pairs.txt"
        
        positive_pairs = [[line.strip().split(",")[0], line.strip().split(",")[1]] 
                         for line in open(positive_pairs_path, "r").readlines()]
        negative_pairs = [[line.strip().split(",")[0], line.strip().split(",")[1]] 
                         for line in open(negative_pairs_path, "r").readlines()]
        
        all_pairs = positive_pairs + negative_pairs
        labels = [1] * len(positive_pairs) + [0] * len(negative_pairs)
        labels_onehot = to_categorical(labels, num_classes=2).astype(np.float32)
        
        datos['labels'] = labels_onehot
        datos['all_pairs'] = all_pairs
        datos['binary_labels'] = np.array(labels)
        
        print(f"  Pares positivos: {len(positive_pairs)}")
        print(f"  Pares negativos: {len(negative_pairs)}")
        print(f"  Total de pares: {len(all_pairs)}")
        
        del positive_pairs, negative_pairs, labels
        gc.collect()
        
    except Exception as e:
        print(f"Error al cargar etiquetas: {e}")
    
    log_memory_usage("Después de cargar datos")
    return datos

def dividir_datos(datos):
    """Divide los datos en conjuntos de entrenamiento, validación y prueba."""
    print("\nDividiendo datos en conjuntos...")
    
    TRAIN_SPLIT = 0.7
    VAL_SPLIT = 0.15
    TEST_SPLIT = 0.15
    
    labels = datos['labels']
    indices = np.arange(labels.shape[0])
    
    train_val_idx, test_idx = train_test_split(
        indices, 
        test_size=TEST_SPLIT, 
        random_state=SEED, 
        stratify=labels.argmax(axis=1)
    )
    
    train_idx, val_idx = train_test_split(
        train_val_idx, 
        test_size=VAL_SPLIT/(1-TEST_SPLIT),
        random_state=SEED, 
        stratify=labels[train_val_idx].argmax(axis=1)
    )
    
    # Crear conjuntos de datos
    train_data = {'labels': labels[train_idx], 'indices': train_idx}
    val_data = {'labels': labels[val_idx], 'indices': val_idx}
    test_data = {'labels': labels[test_idx], 'indices': test_idx}
    
    # Añadir etiquetas binarias
    train_data['binary_labels'] = datos['binary_labels'][train_idx]
    val_data['binary_labels'] = datos['binary_labels'][val_idx]
    test_data['binary_labels'] = datos['binary_labels'][test_idx]
    
    # Añadir pares
    train_data['pairs'] = [datos['all_pairs'][i] for i in train_idx]
    val_data['pairs'] = [datos['all_pairs'][i] for i in val_idx]
    test_data['pairs'] = [datos['all_pairs'][i] for i in test_idx]
    
    # Dividir características
    for nombre, array in datos.items():
        if nombre not in ['labels', 'all_pairs', 'binary_labels']:
            train_data[nombre] = array[train_idx]
            val_data[nombre] = array[val_idx]
            test_data[nombre] = array[test_idx]
    
    # Guardar índices
    np.save(f"{RESULTS_PATH}train_indices.npy", train_idx)
    np.save(f"{RESULTS_PATH}val_indices.npy", val_idx)
    np.save(f"{RESULTS_PATH}test_indices.npy", test_idx)
    
    print(f"Train: {len(train_idx)}, Validación: {len(val_idx)}, Test: {len(test_idx)}")
    
    return train_data, val_data, test_data

def normalizar_datos(train_data, val_data, test_data):
    """Normaliza los datos usando estadísticas del conjunto de entrenamiento."""
    print("\nNormalizando datos...")
    
    train_data_norm = {k: v for k, v in train_data.items()}
    val_data_norm = {k: v for k, v in val_data.items()}
    test_data_norm = {k: v for k, v in test_data.items()}
    
    for nombre in train_data:
        if nombre not in ['labels', 'binary_labels', 'pairs', 'indices']:
            if train_data[nombre].dtype == np.float32:
                mean = np.mean(train_data[nombre], axis=0)
                std = np.std(train_data[nombre], axis=0)
                std = np.where(std < 1e-10, 1.0, std)
                
                # Guardar estadísticas
                np.save(f"{RESULTS_PATH}mean_{nombre}.npy", mean)
                np.save(f"{RESULTS_PATH}std_{nombre}.npy", std)
                
                # Aplicar normalización
                train_data_norm[nombre] = (train_data[nombre] - mean) / std
                val_data_norm[nombre] = (val_data[nombre] - mean) / std
                test_data_norm[nombre] = (test_data[nombre] - mean) / std
                
                print(f"- Normalizado {nombre}")
                del mean, std
                gc.collect()
    
    return train_data_norm, val_data_norm, test_data_norm

def preparar_datos_entrenamiento(data):
    """Prepara los datos para el entrenamiento del modelo."""
    model_inputs = []
    for nombre, array in data.items():
        if nombre not in ['labels', 'binary_labels', 'pairs', 'indices']:
            model_inputs.append(array)
    
    return model_inputs, data['labels']

#------------------------------------------------------------------------------
# Arquitectura del Modelo
#------------------------------------------------------------------------------

def transformer_encoder_block(inputs, head_size, num_heads, ff_dim, dropout=0):
    """Crea un bloque transformer encoder."""
    # Multi-head attention
    attention_output = layers.MultiHeadAttention(
        num_heads=num_heads, key_dim=head_size, dropout=dropout
    )(inputs, inputs)
    # Add & Norm
    attention_output = layers.LayerNormalization(epsilon=1e-6)(inputs + attention_output)
    # Feed Forward
    ff_output = layers.Dense(ff_dim, activation="gelu")(attention_output)
    ff_output = layers.Dense(inputs.shape[-1])(ff_output)
    # Add & Norm
    ff_output = layers.Dropout(dropout)(ff_output)
    return layers.LayerNormalization(epsilon=1e-6)(attention_output + ff_output)

def crear_modelo(datos_caracteristicas, params):
    """
    Crea el modelo de transformer con los parámetros especificados.
    """
    tf.keras.backend.clear_session()
    gc.collect()
    
    print("Creando modelo con parámetros optimizados...")
    log_memory_usage("Antes de crear modelo")
    
    # Extraer parámetros
    embedding_dim = params['embedding_dim']
    num_heads = params['num_heads']
    num_transformer_blocks = params['num_transformer_blocks']
    dropout_rate = params['dropout_rate']
    learning_rate = params['learning_rate']
    weight_decay = params['weight_decay']
    use_regularization = params['use_regularization']
    ff_dim_factor = params['ff_dim_factor']
    cnn_filters1 = params['cnn_filters1']
    cnn_filters2 = params['cnn_filters2']
    cnn_kernel_size = params['cnn_kernel_size']
    cnn_pool_size = params['cnn_pool_size']
    cnn_activation = params['cnn_activation']
    
    # Entradas y proyecciones específicas por tipo de dato
    all_inputs = []
    all_projected_features = []
    
    # Regularización L2 si está habilitada
    l2_reg = keras.regularizers.l2(weight_decay) if use_regularization else None
    
    print("Procesando entradas del modelo:")
    
    # Procesar los datos según su tipo
    for nombre, array in datos_caracteristicas.items():
        if nombre in ['labels', 'binary_labels', 'pairs', 'indices']:
            continue
            
        # Crear capa de entrada
        input_shape = array.shape[1:]
        dtype = array.dtype
        
        input_dtype = 'uint8' if dtype == np.uint8 else 'float32'
        input_layer = keras.Input(shape=input_shape, name=f"input_{nombre}", dtype=input_dtype)
        all_inputs.append(input_layer)
        
        print(f"  - {nombre}: {input_shape} ({input_dtype})")
        
        # Estrategia de procesamiento según el tipo de dato
        if 'embeddings' in nombre:
            # Para embeddings, usar capas densas con regularización
            x = layers.Dense(embedding_dim, activation="relu", 
                    kernel_regularizer=l2_reg)(input_layer)
            x = layers.BatchNormalization()(x)
            x = layers.Dropout(dropout_rate)(x)
            x = layers.Dense(embedding_dim, activation="gelu", 
                    kernel_regularizer=l2_reg)(x)
            all_projected_features.append(layers.BatchNormalization()(x))
        
        elif 'features' in nombre:
            # Para características, una capa más simple
            x = layers.Dense(embedding_dim, activation="gelu", 
                    kernel_regularizer=l2_reg)(input_layer)
            all_projected_features.append(layers.Dropout(dropout_rate)(x))
        
        elif any(s in nombre for s in ['seq', 'mask', '2d', 'normalized']):
            # Para datos de secuencia
            x = layers.Lambda(lambda x: tf.cast(x, tf.float32))(input_layer)
            
            if 'mask' in nombre:
                x = layers.Dense(embedding_dim//2, activation="relu")(x)
                all_projected_features.append(layers.Dense(embedding_dim, activation="gelu")(x))
            else:
                # Para datos de secuencia, usar capas convolucionales
                if len(input_shape) == 1:
                    x = layers.Reshape((input_shape[0], 1))(x)
                
                # Aplicar capas convolucionales con parámetros optimizados
                x = layers.Conv1D(
                    filters=cnn_filters1, 
                    kernel_size=cnn_kernel_size, 
                    activation=cnn_activation, 
                    padding="same"
                )(x)
                x = layers.MaxPooling1D(pool_size=cnn_pool_size)(x)
                x = layers.Conv1D(
                    filters=cnn_filters2, 
                    kernel_size=cnn_kernel_size, 
                    activation=cnn_activation, 
                    padding="same"
                )(x)
                
                # GlobalPooling para obtener un vector por secuencia
                x = layers.GlobalAveragePooling1D()(x)
                x = layers.Dense(embedding_dim, activation="gelu")(x)
                all_projected_features.append(layers.Dropout(dropout_rate)(x))
        
        else:
            # Para cualquier otro tipo de dato
            x = layers.Dense(embedding_dim, activation="gelu")(input_layer)
            all_projected_features.append(layers.Dropout(dropout_rate/2)(x))
    
    print(f"Total de entradas procesadas: {len(all_inputs)}")
    
    # Apilar características proyectadas para crear la secuencia para el transformer
    sequence = layers.Lambda(lambda x: tf.stack(x, axis=1))(all_projected_features)
    
    # Añadir embedding posicional
    position_embedding = layers.Embedding(
        input_dim=len(all_projected_features), 
        output_dim=embedding_dim
    )(tf.range(start=0, limit=len(all_projected_features), delta=1))
    
    sequence = sequence + position_embedding
    
    # Aplicar bloques transformer
    print(f"Aplicando {num_transformer_blocks} bloques transformer...")
    for i in range(num_transformer_blocks):
        sequence = transformer_encoder_block(
            sequence, 
            head_size=embedding_dim // num_heads,
            num_heads=num_heads,
            ff_dim=embedding_dim * ff_dim_factor,
            dropout=dropout_rate
        )
    
    # Global pooling para obtener una representación única
    sequence_features = layers.GlobalAveragePooling1D()(sequence)
    
    # Capas finales de clasificación con capas residuales
    x = layers.Dense(embedding_dim, activation="gelu")(sequence_features)
    x = layers.Dropout(dropout_rate)(x)
    
    # Conexión residual
    x = layers.Add()([x, layers.Dense(embedding_dim, activation=None)(sequence_features)])
    x = layers.LayerNormalization(epsilon=1e-6)(x)
    
    x = layers.Dense(embedding_dim // 2, activation="gelu")(x)
    x = layers.Dropout(dropout_rate/2)(x)
    
    # Capa de salida
    outputs = layers.Dense(2, activation="softmax", name="output")(x)
    
    # Crear modelo
    model = keras.Model(inputs=all_inputs, outputs=outputs)
    
    # Compilar modelo
    optimizer = keras.optimizers.Adam(learning_rate=learning_rate)
    
    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=["accuracy", 
                 keras.metrics.AUC(name="auc"),
                 keras.metrics.Precision(name="precision"),
                 keras.metrics.Recall(name="recall"),
                 keras.metrics.F1Score(name="f1_score", average="weighted")]
    )
    
    # Imprimir resumen del modelo
    print("\n=== RESUMEN DEL MODELO ===")
    model.summary()
    
    # Contar parámetros
    trainable_params = sum(np.prod(v.shape) for v in model.trainable_weights)
    non_trainable_params = sum(np.prod(v.shape) for v in model.non_trainable_weights)
    total_params = trainable_params + non_trainable_params
    
    print(f"\nParámetros entrenables: {trainable_params:,}")
    print(f"Parámetros no entrenables: {non_trainable_params:,}")
    print(f"Total de parámetros: {total_params:,}")
    
    log_memory_usage("Después de crear modelo")
    return model

#------------------------------------------------------------------------------
# Visualización de la Arquitectura del Modelo
#------------------------------------------------------------------------------

def visualizar_arquitectura_modelo(model, save_path):
    """Crea visualizaciones detalladas de la arquitectura del modelo."""
    print("\n=== GENERANDO VISUALIZACIONES DE LA ARQUITECTURA ===")
    
    try:
        # 1. Gráfico básico del modelo
        tf.keras.utils.plot_model(
            model, 
            to_file=f"{save_path}/model_architecture_basic.png",
            show_shapes=True, 
            show_dtype=True,
            show_layer_names=True, 
            rankdir='TB',
            expand_nested=False,
            dpi=150
        )
        print("✅ Gráfico básico de arquitectura guardado")
        
        # 2. Gráfico expandido del modelo
        tf.keras.utils.plot_model(
            model, 
            to_file=f"{save_path}/model_architecture_expanded.png",
            show_shapes=True, 
            show_dtype=True,
            show_layer_names=True, 
            rankdir='TB',
            expand_nested=True,
            dpi=150
        )
        print("✅ Gráfico expandido de arquitectura guardado")
        
        # 3. Análisis de capas por tipo
        layer_types = {}
        layer_params = {}
        for layer in model.layers:
            layer_type = layer.__class__.__name__
            if layer_type not in layer_types:
                layer_types[layer_type] = 0
                layer_params[layer_type] = 0
            layer_types[layer_type] += 1
            if len(layer.weights) > 0:
                layer_params[layer_type] += sum(np.prod(w.shape) for w in layer.weights)
        
        # Visualizar distribución de tipos de capas
        plt.figure(figsize=(15, 10))
        
        plt.subplot(2, 2, 1)
        plt.pie(layer_types.values(), labels=layer_types.keys(), autopct='%1.1f%%')
        plt.title('Distribución de Tipos de Capas')
        
        plt.subplot(2, 2, 2)
        layer_names = list(layer_types.keys())
        layer_counts = list(layer_types.values())
        plt.bar(layer_names, layer_counts)
        plt.title('Número de Capas por Tipo')
        plt.xticks(rotation=45)
        
        plt.subplot(2, 2, 3)
        param_types = [k for k, v in layer_params.items() if v > 0]
        param_counts = [v for k, v in layer_params.items() if v > 0]
        plt.bar(param_types, param_counts)
        plt.title('Parámetros por Tipo de Capa')
        plt.xticks(rotation=45)
        plt.yscale('log')
        
        plt.subplot(2, 2, 4)
        # Información detallada de capas críticas
        critical_layers = []
        for i, layer in enumerate(model.layers):
            if len(layer.weights) > 0:
                param_count = sum(np.prod(w.shape) for w in layer.weights)
                if param_count > 1000:  # Solo capas con más de 1000 parámetros
                    critical_layers.append({
                        'name': layer.name,
                        'type': layer.__class__.__name__,
                        'params': param_count,
                        'position': i
                    })
        
        if critical_layers:
            critical_df = pd.DataFrame(critical_layers)
            top_layers = critical_df.nlargest(10, 'params')
            plt.barh(range(len(top_layers)), top_layers['params'])
            plt.yticks(range(len(top_layers)), [f"{row['type']}\n{row['name']}" for _, row in top_layers.iterrows()])
            plt.title('Top 10 Capas por Número de Parámetros')
            plt.xscale('log')
        
        plt.tight_layout()
        plt.savefig(f"{save_path}/model_layer_analysis.png", dpi=150, bbox_inches='tight')
        plt.close()
        print("✅ Análisis de capas guardado")
        
        # 4. Guardar información detallada en texto
        with open(f"{save_path}/model_architecture_details.txt", "w") as f:
            f.write("=== DETALLES DE LA ARQUITECTURA DEL MODELO ===\n\n")
            
            f.write("=== PARÁMETROS DEL MODELO ===\n")
            for param, value in OPTIMIZED_PARAMS.items():
                f.write(f"{param}: {value}\n")
            f.write("\n")
            
            f.write("=== RESUMEN DEL MODELO ===\n")
            model.summary(print_fn=lambda x: f.write(x + '\n'))
            f.write("\n")
            
            f.write("=== DISTRIBUCIÓN DE TIPOS DE CAPAS ===\n")
            for layer_type, count in layer_types.items():
                f.write(f"{layer_type}: {count} capas\n")
            f.write("\n")
            
            f.write("=== PARÁMETROS POR TIPO DE CAPA ===\n")
            for layer_type, params in layer_params.items():
                if params > 0:
                    f.write(f"{layer_type}: {params:,} parámetros\n")
            f.write("\n")
            
            f.write("=== CAPAS CRÍTICAS (>1000 parámetros) ===\n")
            for layer_info in sorted(critical_layers, key=lambda x: x['params'], reverse=True):
                f.write(f"{layer_info['name']} ({layer_info['type']}): {layer_info['params']:,} parámetros\n")
        
        print("✅ Detalles de arquitectura guardados en texto")
        
    except Exception as e:
        print(f"Error al generar visualizaciones de arquitectura: {e}")
        traceback.print_exc()

#------------------------------------------------------------------------------
# Callbacks y Entrenamiento
#------------------------------------------------------------------------------

class DetailedWandbCallback(tf.keras.callbacks.Callback):
    """Callback detallado para W&B con análisis exhaustivo."""
    
    def __init__(self, validation_data=None):
        super().__init__()
        self.validation_data = validation_data
        self.best_val_loss = float('inf')
        self.best_epoch = 0
        
    def on_epoch_end(self, epoch, logs=None):
        if not WANDB_AVAILABLE:
            return
            
        logs = logs or {}
        
        try:
            # Registrar métricas básicas
            wandb.log(logs)
            
            # Registrar learning rate
            if hasattr(self.model.optimizer, 'lr'):
                lr = self.model.optimizer.lr
                if hasattr(lr, 'numpy'):
                    lr = lr.numpy()
                wandb.log({"learning_rate": float(lr)})
            
            # Análisis de overfitting
            if 'val_loss' in logs and 'loss' in logs:
                overfitting_ratio = logs['val_loss'] / max(logs['loss'], 1e-7)
                wandb.log({"overfitting_ratio": overfitting_ratio})
            
            # Actualizar mejor época
            if 'val_loss' in logs and logs['val_loss'] < self.best_val_loss:
                self.best_val_loss = logs['val_loss']
                self.best_epoch = epoch
                wandb.log({"best_val_loss": self.best_val_loss, "best_epoch": self.best_epoch})
            
            # Cada 50 épocas, hacer análisis más detallado
            if epoch % 50 == 0 and self.validation_data is not None:
                self._detailed_analysis(epoch)
                
        except Exception as e:
            print(f"Error en callback de W&B: {e}")
    
    def _detailed_analysis(self, epoch):
        """Análisis detallado cada cierto número de épocas."""
        try:
            # Obtener predicciones en validación
            val_inputs, val_labels = self.validation_data
            predictions = self.model.predict(val_inputs, verbose=0)
            
            # Calcular métricas adicionales
            y_true = np.argmax(val_labels, axis=1)
            y_pred = np.argmax(predictions, axis=1)
            y_prob = predictions[:, 1]
            
            # AUC ROC
            auc_score = roc_auc_score(y_true, y_prob)
            
            # Average Precision
            avg_precision = average_precision_score(y_true, y_prob)
            
            # Registrar métricas
            wandb.log({
                f"detailed_auc_epoch_{epoch}": auc_score,
                f"detailed_avg_precision_epoch_{epoch}": avg_precision,
                "epoch": epoch
            })
            
        except Exception as e:
            print(f"Error en análisis detallado: {e}")

def entrenar_modelo(model, train_data, val_data, params):
    """Entrena el modelo con los parámetros especificados."""
    print("\n=== INICIANDO ENTRENAMIENTO DEL MODELO ===")
    log_memory_usage("Antes de entrenamiento")
    
    # Preparar datos
    train_inputs, train_labels = preparar_datos_entrenamiento(train_data)
    val_inputs, val_labels = preparar_datos_entrenamiento(val_data)
    
    print(f"Datos de entrenamiento: {len(train_inputs[0])} muestras")
    print(f"Datos de validación: {len(val_inputs[0])} muestras")
    
    # Inicializar W&B si está disponible
    if WANDB_AVAILABLE:
        try:
            wandb.init(
                project="miRNA-lncRNA-Final",
                name="final_optimized_model",
                config=params
            )
        except Exception as e:
            print(f"Error al inicializar W&B: {e}")
    
    # Crear callbacks
    callbacks = [
        # Early stopping
        keras.callbacks.EarlyStopping(
            monitor='val_loss',
            patience=params['patience_early'],
            restore_best_weights=True,
            verbose=1,
            mode='min'
        ),
        
        # Reducción del learning rate
        keras.callbacks.ReduceLROnPlateau(
            monitor='val_loss',
            factor=0.5,
            patience=params['patience_reduce_lr'],
            min_lr=1e-7,
            verbose=1
        ),
        
        # Guardar el mejor modelo
        keras.callbacks.ModelCheckpoint(
            filepath=f"{RESULTS_PATH}best_model.keras",
            monitor='val_loss',
            save_best_only=True,
            verbose=1,
            mode='min'
        ),
        
        # CSV Logger
        keras.callbacks.CSVLogger(
            filename=f"{RESULTS_PATH}training_log.csv",
            separator=',',
            append=False
        ),
        
        # Callback personalizado para W&B
        DetailedWandbCallback(validation_data=(val_inputs, val_labels)) if WANDB_AVAILABLE else keras.callbacks.Callback()
    ]
    
    # Entrenar modelo
    print(f"Entrenando por máximo {params['epochs']} épocas con batch size {params['batch_size']}")
    
    start_time = time.time()
    
    history = model.fit(
        train_inputs, train_labels,
        validation_data=(val_inputs, val_labels),
        epochs=params['epochs'],
        batch_size=params['batch_size'],
        callbacks=callbacks,
        verbose=1
    )
    
    training_time = time.time() - start_time
    print(f"\nEntrenamiento completado en {training_time/3600:.2f} horas")
    
    # Guardar historia
    history_df = pd.DataFrame(history.history)
    history_df.to_csv(f"{RESULTS_PATH}training_history.csv", index=False)
    
    # Guardar modelo final
    model.save(f"{RESULTS_PATH}final_model.keras")
    
    log_memory_usage("Después de entrenamiento")
    return history, model

#------------------------------------------------------------------------------
# Evaluación Exhaustiva con Exportación CSV
#------------------------------------------------------------------------------

def evaluar_todos_los_datos(model, train_data, val_data, test_data):
    """Evalúa el modelo en todos los conjuntos de datos y genera Excel completo + archivos Excel con salidas softmax."""
    print("\n=== EVALUACIÓN EXHAUSTIVA DE TODOS LOS DATOS ===")
    
    results = {}
    all_data = []
    
    # Evaluar cada conjunto
    for dataset_name, data in [("train", train_data), ("validation", val_data), ("test", test_data)]:
        print(f"\nEvaluando conjunto {dataset_name}...")
        
        # Preparar datos
        inputs, labels = preparar_datos_entrenamiento(data)
        
        # Obtener predicciones
        predictions = model.predict(inputs, batch_size=1024, verbose=1)
        
        # Calcular métricas
        y_true = np.argmax(labels, axis=1)
        y_pred = np.argmax(predictions, axis=1)
        y_prob = predictions[:, 1]
        
        # Métricas básicas
        accuracy = accuracy_score(y_true, y_pred)
        precision = precision_score(y_true, y_pred)
        recall = recall_score(y_true, y_pred)
        f1 = f1_score(y_true, y_pred)
        auc_score = roc_auc_score(y_true, y_prob)
        avg_precision = average_precision_score(y_true, y_prob)
        
        results[dataset_name] = {
            'accuracy': accuracy,
            'precision': precision,
            'recall': recall,
            'f1': f1,
            'auc': auc_score,
            'avg_precision': avg_precision,
            'n_samples': len(y_true),
            'n_positive': sum(y_true),
            'n_negative': len(y_true) - sum(y_true)
        }
        
        print(f"  Accuracy: {accuracy:.4f}")
        print(f"  Precision: {precision:.4f}")
        print(f"  Recall: {recall:.4f}")
        print(f"  F1: {f1:.4f}")
        print(f"  AUC: {auc_score:.4f}")
        
        # Crear DataFrame para este conjunto
        for i in range(len(y_true)):
            row = {
                'dataset': dataset_name,
                'sample_index': data['indices'][i],
                'mirna': data['pairs'][i][0],
                'lncrna': data['pairs'][i][1],
                'true_label': y_true[i],
                'predicted_label': y_pred[i],
                'probability_negative': predictions[i][0],
                'probability_positive': predictions[i][1],
                'correct_prediction': y_true[i] == y_pred[i]
            }
            all_data.append(row)
    
    # Crear DataFrame completo
    df_complete = pd.DataFrame(all_data)
    
    # *** NUEVA FUNCIONALIDAD: Guardar XLSX con salidas del softmax ***
    print(f"\n=== GUARDANDO ARCHIVOS EXCEL CON SALIDAS DEL SOFTMAX ===")
    
    # Crear DataFrame específico para las salidas del softmax
    df_softmax_outputs = df_complete[[
        'dataset', 'sample_index', 'mirna', 'lncrna',
        'true_label', 'predicted_label', 
        'probability_negative', 'probability_positive',
        'correct_prediction'
    ]].copy()
    
    # Renombrar columnas para mayor claridad
    df_softmax_outputs = df_softmax_outputs.rename(columns={
        'true_label': 'real_label',
        'predicted_label': 'model_prediction',
        'probability_negative': 'softmax_output_negative',
        'probability_positive': 'softmax_output_positive'
    })
    
    # Añadir columnas adicionales útiles
    df_softmax_outputs['prediction_confidence'] = df_softmax_outputs[['softmax_output_negative', 'softmax_output_positive']].max(axis=1)
    df_softmax_outputs['prediction_uncertainty'] = 1 - df_softmax_outputs['prediction_confidence']
    df_softmax_outputs['softmax_difference'] = abs(df_softmax_outputs['softmax_output_positive'] - df_softmax_outputs['softmax_output_negative'])
    
    # Guardar archivo Excel principal con todas las salidas del softmax
    xlsx_filename = f"{RESULTS_PATH}softmax_outputs_all_data.xlsx"
    df_softmax_outputs.to_excel(xlsx_filename, index=False, engine='openpyxl')
    print(f"✅ Excel con salidas del softmax guardado: {xlsx_filename}")
    
    # Guardar archivos Excel separados por conjunto de datos
    for dataset in ['train', 'validation', 'test']:
        df_dataset = df_softmax_outputs[df_softmax_outputs['dataset'] == dataset]
        xlsx_dataset_filename = f"{RESULTS_PATH}softmax_outputs_{dataset}.xlsx"
        df_dataset.to_excel(xlsx_dataset_filename, index=False, engine='openpyxl')
        print(f"✅ Excel {dataset} guardado: {xlsx_dataset_filename}")
    
    # Crear resumen estadístico de las salidas del softmax
    softmax_summary = []
    for dataset in ['train', 'validation', 'test']:
        df_dataset = df_softmax_outputs[df_softmax_outputs['dataset'] == dataset]
        
        summary_stats = {
            'dataset': dataset,
            'n_samples': len(df_dataset),
            'n_positive_real': len(df_dataset[df_dataset['real_label'] == 1]),
            'n_negative_real': len(df_dataset[df_dataset['real_label'] == 0]),
            'mean_prob_positive': df_dataset['softmax_output_positive'].mean(),
            'std_prob_positive': df_dataset['softmax_output_positive'].std(),
            'mean_prob_negative': df_dataset['softmax_output_negative'].mean(),
            'std_prob_negative': df_dataset['softmax_output_negative'].std(),
            'mean_confidence': df_dataset['prediction_confidence'].mean(),
            'mean_uncertainty': df_dataset['prediction_uncertainty'].mean(),
            'accuracy': df_dataset['correct_prediction'].mean()
        }
        softmax_summary.append(summary_stats)
    
    df_softmax_summary = pd.DataFrame(softmax_summary)
    df_softmax_summary.to_excel(f"{RESULTS_PATH}softmax_outputs_summary.xlsx", index=False, engine='openpyxl')
    print(f"✅ Resumen de salidas del softmax guardado: {RESULTS_PATH}softmax_outputs_summary.xlsx")
    
    # Crear archivo Excel consolidado con múltiples hojas
    xlsx_consolidated_filename = f"{RESULTS_PATH}softmax_outputs_consolidated.xlsx"
    with pd.ExcelWriter(xlsx_consolidated_filename, engine='openpyxl') as writer:
        # Hoja con todos los datos
        df_softmax_outputs.to_excel(writer, sheet_name='All_Data', index=False)
        
        # Hojas separadas por conjunto
        for dataset in ['train', 'validation', 'test']:
            df_dataset = df_softmax_outputs[df_softmax_outputs['dataset'] == dataset]
            df_dataset.to_excel(writer, sheet_name=f'{dataset.capitalize()}_Data', index=False)
        
        # Hoja de resumen
        df_softmax_summary.to_excel(writer, sheet_name='Summary_Statistics', index=False)
        
        # Hoja con solo errores (predicciones incorrectas)
        df_errors_softmax = df_softmax_outputs[df_softmax_outputs['correct_prediction'] == False]
        df_errors_softmax.to_excel(writer, sheet_name='Prediction_Errors', index=False)
        
        # Hoja con análisis de confianza
        confidence_analysis = []
        for dataset in ['train', 'validation', 'test']:
            df_dataset = df_softmax_outputs[df_softmax_outputs['dataset'] == dataset]
            
            # Análisis por cuartiles de confianza
            quartiles = df_dataset['prediction_confidence'].quantile([0.25, 0.5, 0.75])
            
            for i, (q_name, q_value) in enumerate(zip(['Q1', 'Q2', 'Q3', 'Q4'], [0, 0.25, 0.5, 0.75])):
                if i == 0:  # Q1: 0-25%
                    mask = df_dataset['prediction_confidence'] <= quartiles[0.25]
                elif i == 1:  # Q2: 25-50%
                    mask = (df_dataset['prediction_confidence'] > quartiles[0.25]) & (df_dataset['prediction_confidence'] <= quartiles[0.5])
                elif i == 2:  # Q3: 50-75%
                    mask = (df_dataset['prediction_confidence'] > quartiles[0.5]) & (df_dataset['prediction_confidence'] <= quartiles[0.75])
                else:  # Q4: 75-100%
                    mask = df_dataset['prediction_confidence'] > quartiles[0.75]
                
                subset = df_dataset[mask]
                if len(subset) > 0:
                    confidence_analysis.append({
                        'dataset': dataset,
                        'confidence_quartile': q_name,
                        'n_samples': len(subset),
                        'accuracy': subset['correct_prediction'].mean(),
                        'avg_confidence': subset['prediction_confidence'].mean(),
                        'avg_uncertainty': subset['prediction_uncertainty'].mean(),
                        'avg_softmax_positive': subset['softmax_output_positive'].mean(),
                        'avg_softmax_negative': subset['softmax_output_negative'].mean()
                    })
        
        df_confidence_analysis = pd.DataFrame(confidence_analysis)
        df_confidence_analysis.to_excel(writer, sheet_name='Confidence_Analysis', index=False)
    
    print(f"✅ Excel consolidado guardado: {xlsx_consolidated_filename}")
    
    # Guardar en Excel con múltiples hojas (funcionalidad original)
    with pd.ExcelWriter(f"{RESULTS_PATH}complete_evaluation_results.xlsx", engine='openpyxl') as writer:
        # Hoja principal con todos los datos
        df_complete.to_excel(writer, sheet_name='All_Data', index=False)
        
        # Hoja separada por conjunto
        for dataset in ['train', 'validation', 'test']:
            df_subset = df_complete[df_complete['dataset'] == dataset]
            df_subset.to_excel(writer, sheet_name=f'{dataset.capitalize()}_Data', index=False)
        
        # Hoja de resumen de métricas
        df_metrics = pd.DataFrame(results).T
        df_metrics.to_excel(writer, sheet_name='Metrics_Summary')
        
        # Hoja de errores (predicciones incorrectas)
        df_errors = df_complete[df_complete['correct_prediction'] == False]
        df_errors.to_excel(writer, sheet_name='Prediction_Errors', index=False)
        
        # Nueva hoja: Salidas del Softmax
        df_softmax_outputs.to_excel(writer, sheet_name='Softmax_Outputs', index=False)
        
        # Nueva hoja: Resumen de Softmax
        df_softmax_summary.to_excel(writer, sheet_name='Softmax_Summary', index=False)
        
        # Hoja de análisis por umbral
        threshold_analysis = []
        for threshold in np.arange(0.1, 1.0, 0.1):
            for dataset in ['train', 'validation', 'test']:
                df_subset = df_complete[df_complete['dataset'] == dataset]
                y_pred_thresh = (df_subset['probability_positive'] >= threshold).astype(int)
                y_true_subset = df_subset['true_label']
                
                acc = accuracy_score(y_true_subset, y_pred_thresh)
                prec = precision_score(y_true_subset, y_pred_thresh, zero_division=0)
                rec = recall_score(y_true_subset, y_pred_thresh, zero_division=0)
                f1_thresh = f1_score(y_true_subset, y_pred_thresh, zero_division=0)
                
                threshold_analysis.append({
                    'dataset': dataset,
                    'threshold': threshold,
                    'accuracy': acc,
                    'precision': prec,
                    'recall': rec,
                    'f1_score': f1_thresh
                })
        
        df_threshold = pd.DataFrame(threshold_analysis)
        df_threshold.to_excel(writer, sheet_name='Threshold_Analysis', index=False)
    
    print(f"\n✅ Resultados completos guardados en: {RESULTS_PATH}complete_evaluation_results.xlsx")
    print(f"✅ Archivos Excel con salidas del softmax:")
    print(f"   - Todos los datos: {RESULTS_PATH}softmax_outputs_all_data.xlsx")
    print(f"   - Por conjunto: {RESULTS_PATH}softmax_outputs_train.xlsx, validation.xlsx, test.xlsx")
    print(f"   - Resumen: {RESULTS_PATH}softmax_outputs_summary.xlsx")
    print(f"   - Consolidado (múltiples hojas): {RESULTS_PATH}softmax_outputs_consolidated.xlsx")
    
    return results, df_complete

def crear_matrices_confusion(df_complete, save_path):
    """Crea matrices de confusión detalladas con cantidades y porcentajes."""
    print("\n=== CREANDO MATRICES DE CONFUSIÓN ===")
    
    datasets = ['train', 'validation', 'test']
    
    # Crear figura con subplots para todas las matrices
    fig, axes = plt.subplots(2, 3, figsize=(20, 12))
    
    for i, dataset in enumerate(datasets):
        df_subset = df_complete[df_complete['dataset'] == dataset]
        y_true = df_subset['true_label']
        y_pred = df_subset['predicted_label']
        
        # Matriz de confusión con cantidades
        cm_counts = confusion_matrix(y_true, y_pred)
        
        # Matriz de confusión con porcentajes
        cm_percentages = confusion_matrix(y_true, y_pred, normalize='true') * 100
        
        # Subplot para cantidades
        ax1 = axes[0, i]
        sns.heatmap(cm_counts, annot=True, fmt='d', cmap='Blues', ax=ax1,
                   xticklabels=['Negativo', 'Positivo'],
                   yticklabels=['Negativo', 'Positivo'])
        ax1.set_title(f'{dataset.capitalize()} - Cantidades')
        ax1.set_xlabel('Predicción')
        ax1.set_ylabel('Verdadero')
        
        # Subplot para porcentajes
        ax2 = axes[1, i]
        sns.heatmap(cm_percentages, annot=True, fmt='.1f', cmap='Reds', ax=ax2,
                   xticklabels=['Negativo', 'Positivo'],
                   yticklabels=['Negativo', 'Positivo'])
        ax2.set_title(f'{dataset.capitalize()} - Porcentajes (%)')
        ax2.set_xlabel('Predicción')
        ax2.set_ylabel('Verdadero')
        
        # Guardar matrices individuales también
        plt.figure(figsize=(10, 8))
        plt.subplot(1, 2, 1)
        sns.heatmap(cm_counts, annot=True, fmt='d', cmap='Blues',
                   xticklabels=['Negativo', 'Positivo'],
                   yticklabels=['Negativo', 'Positivo'])
        plt.title(f'{dataset.capitalize()} - Cantidades')
        plt.xlabel('Predicción')
        plt.ylabel('Verdadero')
        
        plt.subplot(1, 2, 2)
        sns.heatmap(cm_percentages, annot=True, fmt='.1f', cmap='Reds',
                   xticklabels=['Negativo', 'Positivo'],
                   yticklabels=['Negativo', 'Positivo'])
        plt.title(f'{dataset.capitalize()} - Porcentajes (%)')
        plt.xlabel('Predicción')
        plt.ylabel('Verdadero')
        
        plt.tight_layout()
        plt.savefig(f"{save_path}/confusion_matrix_{dataset}.png", dpi=300, bbox_inches='tight')
        plt.close()
        
        # Imprimir métricas detalladas
        tn, fp, fn, tp = cm_counts.ravel()
        print(f"\n{dataset.capitalize()} Dataset:")
        print(f"  True Negatives: {tn}")
        print(f"  False Positives: {fp}")
        print(f"  False Negatives: {fn}")
        print(f"  True Positives: {tp}")
        print(f"  Sensitivity (Recall): {tp/(tp+fn):.4f}")
        print(f"  Specificity: {tn/(tn+fp):.4f}")
        print(f"  Precision: {tp/(tp+fp):.4f}")
    
    plt.tight_layout()
    fig.savefig(f"{save_path}/confusion_matrices_all.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✅ Matrices de confusión creadas y guardadas")

#------------------------------------------------------------------------------
# Generación de Gráficos Exhaustivos
#------------------------------------------------------------------------------

def generar_graficos_exhaustivos(history, results, df_complete, save_path):
    """Genera una amplia variedad de gráficos de análisis."""
    print("\n=== GENERANDO GRÁFICOS EXHAUSTIVOS ===")
    
    # 1. Métricas de entrenamiento
    print("Generando gráficos de entrenamiento...")
    
    metrics = ['loss', 'accuracy', 'auc', 'precision', 'recall', 'f1_score']
    
    plt.figure(figsize=(20, 15))
    for i, metric in enumerate(metrics):
        if metric in history.history:
            plt.subplot(2, 3, i+1)
            plt.plot(history.history[metric], label=f'Train {metric.title()}')
            if f'val_{metric}' in history.history:
                plt.plot(history.history[f'val_{metric}'], label=f'Val {metric.title()}')
            plt.title(f'{metric.title()} Durante el Entrenamiento')
            plt.xlabel('Época')
            plt.ylabel(metric.title())
            plt.legend()
            plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/training_metrics.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 2. Análisis de overfitting
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 3, 1)
    plt.plot(history.history['loss'], label='Train Loss')
    plt.plot(history.history['val_loss'], label='Val Loss')
    plt.title('Análisis de Overfitting - Loss')
    plt.xlabel('Época')
    plt.ylabel('Loss')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 2)
    loss_ratio = np.array(history.history['val_loss']) / np.array(history.history['loss'])
    plt.plot(loss_ratio, label='Val Loss / Train Loss')
    plt.axhline(y=1, color='r', linestyle='--', label='Ratio = 1')
    plt.title('Ratio de Overfitting')
    plt.xlabel('Época')
    plt.ylabel('Ratio')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(1, 3, 3)
    loss_diff = np.array(history.history['val_loss']) - np.array(history.history['loss'])
    plt.plot(loss_diff, label='Val Loss - Train Loss')
    plt.axhline(y=0, color='r', linestyle='--', label='Diferencia = 0')
    plt.title('Diferencia de Loss')
    plt.xlabel('Época')
    plt.ylabel('Diferencia')
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/overfitting_analysis.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 3. Curvas ROC para cada conjunto
    plt.figure(figsize=(15, 5))
    
    for i, dataset in enumerate(['train', 'validation', 'test']):
        plt.subplot(1, 3, i+1)
        df_subset = df_complete[df_complete['dataset'] == dataset]
        y_true = df_subset['true_label']
        y_prob = df_subset['probability_positive']
        
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        
        plt.plot(fpr, tpr, color='darkorange', lw=2, 
                label=f'ROC curve (AUC = {roc_auc:.3f})')
        plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('Tasa de Falsos Positivos')
        plt.ylabel('Tasa de Verdaderos Positivos')
        plt.title(f'Curva ROC - {dataset.capitalize()}')
        plt.legend(loc="lower right")
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/roc_curves.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 4. Curvas Precision-Recall
    plt.figure(figsize=(15, 5))
    
    for i, dataset in enumerate(['train', 'validation', 'test']):
        plt.subplot(1, 3, i+1)
        df_subset = df_complete[df_complete['dataset'] == dataset]
        y_true = df_subset['true_label']
        y_prob = df_subset['probability_positive']
        
        precision_curve, recall_curve, _ = precision_recall_curve(y_true, y_prob)
        avg_precision = average_precision_score(y_true, y_prob)
        baseline = sum(y_true) / len(y_true)
        
        plt.plot(recall_curve, precision_curve, color='blue', lw=2,
                label=f'PR curve (AP = {avg_precision:.3f})')
        plt.axhline(y=baseline, color='red', linestyle='--', 
                   label=f'Baseline = {baseline:.3f}')
        plt.xlim([0.0, 1.0])
        plt.ylim([0.0, 1.05])
        plt.xlabel('Recall')
        plt.ylabel('Precision')
        plt.title(f'Curva Precision-Recall - {dataset.capitalize()}')
        plt.legend(loc="best")
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/precision_recall_curves.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 5. Distribución de probabilidades
    plt.figure(figsize=(15, 10))
    
    for i, dataset in enumerate(['train', 'validation', 'test']):
        df_subset = df_complete[df_complete['dataset'] == dataset]
        
        plt.subplot(2, 3, i+1)
        # Histograma por clase
        pos_probs = df_subset[df_subset['true_label'] == 1]['probability_positive']
        neg_probs = df_subset[df_subset['true_label'] == 0]['probability_positive']
        
        plt.hist(neg_probs, bins=50, alpha=0.7, label='Negativo', color='red')
        plt.hist(pos_probs, bins=50, alpha=0.7, label='Positivo', color='blue')
        plt.xlabel('Probabilidad Positiva')
        plt.ylabel('Frecuencia')
        plt.title(f'Distribución de Probabilidades - {dataset.capitalize()}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        plt.subplot(2, 3, i+4)
        # Box plot
        prob_data = [neg_probs, pos_probs]
        plt.boxplot(prob_data, labels=['Negativo', 'Positivo'])
        plt.ylabel('Probabilidad Positiva')
        plt.title(f'Box Plot - {dataset.capitalize()}')
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/probability_distributions.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 6. Análisis de umbrales
    plt.figure(figsize=(20, 15))
    
    thresholds = np.arange(0.1, 1.0, 0.05)
    
    for i, dataset in enumerate(['train', 'validation', 'test']):
        df_subset = df_complete[df_complete['dataset'] == dataset]
        y_true = df_subset['true_label']
        y_prob = df_subset['probability_positive']
        
        metrics_by_threshold = {
            'accuracy': [],
            'precision': [],
            'recall': [],
            'f1': [],
            'specificity': []
        }
        
        for threshold in thresholds:
            y_pred_thresh = (y_prob >= threshold).astype(int)
            
            # Calcular métricas
            acc = accuracy_score(y_true, y_pred_thresh)
            prec = precision_score(y_true, y_pred_thresh, zero_division=0)
            rec = recall_score(y_true, y_pred_thresh, zero_division=0)
            f1_thresh = f1_score(y_true, y_pred_thresh, zero_division=0)
            
            # Calcular especificidad
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred_thresh).ravel()
            spec = tn / (tn + fp) if (tn + fp) > 0 else 0
            
            metrics_by_threshold['accuracy'].append(acc)
            metrics_by_threshold['precision'].append(prec)
            metrics_by_threshold['recall'].append(rec)
            metrics_by_threshold['f1'].append(f1_thresh)
            metrics_by_threshold['specificity'].append(spec)
        
        # Subplot para este dataset
        plt.subplot(2, 3, i+1)
        for metric, values in metrics_by_threshold.items():
            plt.plot(thresholds, values, 'o-', label=metric.capitalize(), linewidth=2)
        
        plt.xlabel('Umbral')
        plt.ylabel('Valor de Métrica')
        plt.title(f'Métricas por Umbral - {dataset.capitalize()}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Subplot para Youden's J statistic
        plt.subplot(2, 3, i+4)
        youden_j = np.array(metrics_by_threshold['recall']) + np.array(metrics_by_threshold['specificity']) - 1
        plt.plot(thresholds, youden_j, 'o-', color='purple', linewidth=2, label="Youden's J")
        plt.plot(thresholds, metrics_by_threshold['recall'], 'o-', color='blue', label='Sensibilidad')
        plt.plot(thresholds, metrics_by_threshold['specificity'], 'o-', color='red', label='Especificidad')
        
        # Encontrar el mejor umbral
        best_threshold_idx = np.argmax(youden_j)
        best_threshold = thresholds[best_threshold_idx]
        plt.axvline(x=best_threshold, color='black', linestyle='--', 
                   label=f'Mejor umbral: {best_threshold:.2f}')
        
        plt.xlabel('Umbral')
        plt.ylabel('Valor')
        plt.title(f"Índice de Youden - {dataset.capitalize()}")
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/threshold_analysis.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 7. Comparación de métricas entre conjuntos
    plt.figure(figsize=(15, 10))
    
    datasets = ['train', 'validation', 'test']
    metrics_comparison = {
        'accuracy': [results[d]['accuracy'] for d in datasets],
        'precision': [results[d]['precision'] for d in datasets],
        'recall': [results[d]['recall'] for d in datasets],
        'f1': [results[d]['f1'] for d in datasets],
        'auc': [results[d]['auc'] for d in datasets],
        'avg_precision': [results[d]['avg_precision'] for d in datasets]
    }
    
    x = np.arange(len(datasets))
    width = 0.12
    
    plt.subplot(2, 1, 1)
    for i, (metric, values) in enumerate(metrics_comparison.items()):
        plt.bar(x + i*width, values, width, label=metric.replace('_', ' ').title())
    
    plt.xlabel('Conjunto de Datos')
    plt.ylabel('Valor de Métrica')
    plt.title('Comparación de Métricas entre Conjuntos')
    plt.xticks(x + width*2.5, [d.capitalize() for d in datasets])
    plt.legend()
    plt.grid(True, alpha=0.3)
    
    plt.subplot(2, 1, 2)
    # Radar chart
    from math import pi
    
    angles = [n / float(len(metrics_comparison)) * 2 * pi for n in range(len(metrics_comparison))]
    angles += angles[:1]
    
    colors = ['red', 'blue', 'green']
    for i, dataset in enumerate(datasets):
        values = [metrics_comparison[metric][i] for metric in metrics_comparison.keys()]
        values += values[:1]
        
        plt.subplot(2, 1, 2, projection='polar')
        plt.plot(angles, values, 'o-', linewidth=2, label=dataset.capitalize(), color=colors[i])
        plt.fill(angles, values, alpha=0.25, color=colors[i])
    
    plt.xticks(angles[:-1], [m.replace('_', ' ').title() for m in metrics_comparison.keys()])
    plt.ylim(0, 1)
    plt.title('Comparación Radar de Métricas')
    plt.legend(loc='upper right', bbox_to_anchor=(0.1, 0.1))
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/metrics_comparison.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 8. Análisis de errores
    print("Generando análisis de errores...")
    
    plt.figure(figsize=(20, 10))
    
    for i, dataset in enumerate(['train', 'validation', 'test']):
        df_subset = df_complete[df_complete['dataset'] == dataset]
        
        # Errores por tipo
        fp_data = df_subset[(df_subset['true_label'] == 0) & (df_subset['predicted_label'] == 1)]
        fn_data = df_subset[(df_subset['true_label'] == 1) & (df_subset['predicted_label'] == 0)]
        
        plt.subplot(2, 3, i+1)
        if len(fp_data) > 0:
            plt.hist(fp_data['probability_positive'], bins=20, alpha=0.7, 
                    label=f'Falsos Positivos (n={len(fp_data)})', color='red')
        if len(fn_data) > 0:
            plt.hist(fn_data['probability_positive'], bins=20, alpha=0.7, 
                    label=f'Falsos Negativos (n={len(fn_data)})', color='orange')
        
        plt.xlabel('Probabilidad Positiva')
        plt.ylabel('Frecuencia')
        plt.title(f'Distribución de Errores - {dataset.capitalize()}')
        plt.legend()
        plt.grid(True, alpha=0.3)
        
        # Análisis de confianza en errores
        plt.subplot(2, 3, i+4)
        
        if len(fp_data) > 0 and len(fn_data) > 0:
            fp_confidence = np.abs(fp_data['probability_positive'] - 0.5)
            fn_confidence = np.abs(fn_data['probability_positive'] - 0.5)
            
            plt.boxplot([fp_confidence, fn_confidence], 
                       labels=['Falsos Positivos', 'Falsos Negativos'])
            plt.ylabel('Confianza del Error (|prob - 0.5|)')
            plt.title(f'Confianza en Errores - {dataset.capitalize()}')
            plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/error_analysis.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    # 9. Calibración del modelo
    plt.figure(figsize=(15, 5))
    
    for i, dataset in enumerate(['train', 'validation', 'test']):
        plt.subplot(1, 3, i+1)
        df_subset = df_complete[df_complete['dataset'] == dataset]
        y_true = df_subset['true_label']
        y_prob = df_subset['probability_positive']
        
        # Binning para calibración
        n_bins = 10
        bin_boundaries = np.linspace(0, 1, n_bins + 1)
        bin_lowers = bin_boundaries[:-1]
        bin_uppers = bin_boundaries[1:]
        
        bin_accuracies = []
        bin_confidences = []
        bin_counts = []
        
        for bin_lower, bin_upper in zip(bin_lowers, bin_uppers):
            in_bin = (y_prob > bin_lower) & (y_prob <= bin_upper)
            prop_in_bin = in_bin.mean()
            bin_counts.append(in_bin.sum())
            
            if prop_in_bin > 0:
                accuracy_in_bin = y_true[in_bin].mean()
                avg_confidence_in_bin = y_prob[in_bin].mean()
                bin_accuracies.append(accuracy_in_bin)
                bin_confidences.append(avg_confidence_in_bin)
            else:
                bin_accuracies.append(0)
                bin_confidences.append(0)
        
        # Gráfico de calibración
        plt.plot([0, 1], [0, 1], 'k--', label='Perfectamente calibrado')
        plt.plot(bin_confidences, bin_accuracies, 'o-', label='Modelo')
        
        # Barras de error basadas en el número de muestras en cada bin
        bin_sizes = np.array(bin_counts)
        errors = np.sqrt(np.array(bin_accuracies) * (1 - np.array(bin_accuracies)) / np.maximum(bin_sizes, 1))
        plt.errorbar(bin_confidences, bin_accuracies, yerr=errors, fmt='o-', alpha=0.7)
        
        plt.xlabel('Confianza Media')
        plt.ylabel('Precisión')
        plt.title(f'Calibración del Modelo - {dataset.capitalize()}')
        plt.legend()
        plt.grid(True, alpha=0.3)
    
    plt.tight_layout()
    plt.savefig(f"{save_path}/model_calibration.png", dpi=300, bbox_inches='tight')
    plt.close()
    
    print("✅ Todos los gráficos exhaustivos generados")

#------------------------------------------------------------------------------
# Función Principal
#------------------------------------------------------------------------------

def main():
    """Función principal del programa."""
    print("=== MODELO miRNA-lncRNA CON PARÁMETROS OPTIMIZADOS ===")
    log_memory_usage("Inicio del programa")
    
    # Configurar TensorFlow
    os.environ['TF_CPP_MIN_LOG_LEVEL'] = '2'
    
    gpus = tf.config.experimental.list_physical_devices('GPU')
    if gpus:
        try:
            for gpu in gpus:
                tf.config.experimental.set_memory_growth(gpu, True)
            print(f"Configuradas {len(gpus)} GPUs")
        except RuntimeError as e:
            print(f"Error al configurar GPUs: {e}")
    
    try:
        # Crear directorios
        for path in [RESULTS_PATH, f"{RESULTS_PATH}plots", f"{RESULTS_PATH}analysis", f"{RESULTS_PATH}model_analysis"]:
            os.makedirs(path, exist_ok=True)
        
        # Configurar W&B
        if WANDB_AVAILABLE:
            try:
                wandb.login(key="1eb48b0e0e231437bc58d5426eaba063e991a679")
                print("W&B configurado correctamente")
            except:
                print("Error al configurar W&B, continuando sin él")
        
        # Cargar y procesar datos
        print("\n" + "="*50)
        datos = cargar_datos()
        
        train_data, val_data, test_data = dividir_datos(datos)
        train_data_norm, val_data_norm, test_data_norm = normalizar_datos(train_data, val_data, test_data)
        
        # Liberar memoria
        del datos, train_data, val_data, test_data
        cleanup_memory()
        
        # Crear modelo
        print("\n" + "="*50)
        model = crear_modelo(train_data_norm, OPTIMIZED_PARAMS)
        
        # Visualizar arquitectura
        visualizar_arquitectura_modelo(model, f"{RESULTS_PATH}model_analysis")
        
        # Entrenar modelo
        print("\n" + "="*50)
        history, trained_model = entrenar_modelo(model, train_data_norm, val_data_norm, OPTIMIZED_PARAMS)
        
        # Evaluación exhaustiva
        print("\n" + "="*50)
        results, df_complete = evaluar_todos_los_datos(trained_model, train_data_norm, val_data_norm, test_data_norm)
        
        # Crear matrices de confusión
        crear_matrices_confusion(df_complete, f"{RESULTS_PATH}plots")
        
        # Generar gráficos exhaustivos
        generar_graficos_exhaustivos(history, results, df_complete, f"{RESULTS_PATH}plots")
        
        # Guardar resumen final
        with open(f"{RESULTS_PATH}final_summary.txt", "w") as f:
            f.write("=== RESUMEN FINAL DEL MODELO ===\n\n")
            f.write("PARÁMETROS UTILIZADOS:\n")
            for param, value in OPTIMIZED_PARAMS.items():
                f.write(f"  {param}: {value}\n")
            f.write("\n")
            
            f.write("RESULTADOS POR CONJUNTO:\n")
            for dataset, metrics in results.items():
                f.write(f"\n{dataset.upper()}:\n")
                for metric, value in metrics.items():
                    f.write(f"  {metric}: {value:.4f}\n")
        
        print("\n" + "="*50)
        print("🎉 ANÁLISIS COMPLETADO EXITOSAMENTE")
        print(f"📊 Resultados guardados en: {RESULTS_PATH}")
        print(f"📈 Gráficos disponibles en: {RESULTS_PATH}plots/")
        print(f"🏗️ Análisis del modelo en: {RESULTS_PATH}model_analysis/")
        print(f"📋 Excel completo: {RESULTS_PATH}complete_evaluation_results.xlsx")
        print(f"📄 Archivos Excel con salidas del softmax:")
        print(f"   🔹 Todos los datos: {RESULTS_PATH}softmax_outputs_all_data.xlsx")
        print(f"   🔹 Por conjuntos: {RESULTS_PATH}softmax_outputs_[train/validation/test].xlsx")
        print(f"   🔹 Resumen: {RESULTS_PATH}softmax_outputs_summary.xlsx")
        print(f"   🔹 Consolidado: {RESULTS_PATH}softmax_outputs_consolidated.xlsx")
        
        if WANDB_AVAILABLE:
            wandb.finish()
        
        return results
        
    except Exception as e:
        print(f"Error durante la ejecución: {e}")
        traceback.print_exc()
        return None

if __name__ == "__main__":
    # Configurar semillas
    np.random.seed(SEED)
    tf.random.set_seed(SEED)
    
    # Configurar TensorFlow para uso óptimo de memoria
    tf.config.threading.set_inter_op_parallelism_threads(2)
    tf.config.threading.set_intra_op_parallelism_threads(4)
    
    # Ejecutar programa principal
    main()

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /Users/andrescubillovillalobos/.netrc


=== MODELO miRNA-lncRNA CON PARÁMETROS OPTIMIZADOS ===
🧠 Inicio del programa Memory Usage: 544.73 MB


wandb: Currently logged in as: stevenm711 (compgen-inc) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


W&B configurado correctamente

=== CARGANDO TODOS LOS DATOS ===
🧠 Antes de cargar datos Memory Usage: 550.45 MB
Se encontraron 19 archivos .npy


Cargando archivos:   0%|          | 0/19 [00:00<?, ?it/s]

...

 Non-trainable params: 768 (3.00 KB)


Parámetros entrenables: 377,826
Parámetros no entrenables: 768
Total de parámetros: 378,594
🧠 Después de crear modelo Memory Usage: 1743.20 MB

=== GENERANDO VISUALIZACIONES DE LA ARQUITECTURA ===
You must install pydot (`pip install pydot`) for `plot_model` to work.
✅ Gráfico básico de arquitectura guardado
You must install pydot (`pip install pydot`) for `plot_model` to work.
✅ Gráfico expandido de arquitectura guardado
✅ Análisis de capas guardado


✅ Detalles de arquitectura guardados en texto


=== INICIANDO ENTRENAMIENTO DEL MODELO ===
🧠 Antes de entrenamiento Memory Usage: 1780.36 MB
Datos de entrenamiento: 32183 muestras
Datos de validación: 6897 muestras


Entrenando por máximo 1000 épocas con batch size 2048
Epoch 1/1000
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 620ms/step - accuracy: 0.5601 - auc: 0.5773 - f1_score: 0.5578 - loss: 1.1231 - precision: 0.5601 - recall: 0.5601
Epoch 1: val_loss improved from None to 0.73677, saving model to ./outputs/best_model.keras

Epoch 1: finished saving model to ./outputs/best_model.keras
16/16 ━━━━━━━━━━━━━━━━━━━━ 18s 798ms/step - accuracy: 0.6208 - auc: 0.6589 - f1_score: 0.6208 - loss: 1.0019 - precision: 0.6208 - recall: 0.6208 - val_accuracy: 0.8369 - val_auc: 0.8766 - val_f1_score: 0.8356 - val_loss: 0.7368 - val_precision: 0.8369 - val_recall: 0.8369 - learning_rate: 0.0010
Epoch 2/1000
16/16 ━━━━━━━━━━━━━━━━━━━━ 0s 607ms/step - accuracy: 0.7907 - auc: 0.8555 - f1_score: 0.7902 - loss: 0.7706 - precision: 0.7907 - recall: 0.7907
Epoch 2: val_loss improved from 0.73677 to 0.59921, saving model to ./outputs/best_model.keras

Epoch 2: finished saving model to ./outputs/best_model.keras
16/16 ━━━━━━━━━━━━━━━

Traceback (most recent call last):
  File "/var/folders/g_/nrm32vwd4y1_x74xcr5ny0mm0000gn/T/ipykernel_2973/2957945924.py", line 1571, in main
    results, df_complete = evaluar_todos_los_datos(trained_model, train_data_norm, val_data_norm, test_data_norm)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/var/folders/g_/nrm32vwd4y1_x74xcr5ny0mm0000gn/T/ipykernel_2973/2957945924.py", line 917, in evaluar_todos_los_datos
    df_softmax_outputs.to_excel(xlsx_filename, index=False, engine='openpyxl')
  File "/opt/homebrew/Caskroom/miniforge/base/envs/compgen/lib/python3.11/site-packages/pandas/core/generic.py", line 2312, in to_excel
    formatter.write(
  File "/opt/homebrew/Caskroom/miniforge/base/envs/compgen/lib/python3.11/site-packages/pandas/io/formats/excel.py", line 1003, in write
    writer = ExcelWriter(
             ^^^^^^^^^^^^
  File "/opt/homebrew/Caskroom/miniforge/base/envs/compgen/lib/python3.11/site-p

In [2]:
import numpy as np

# Cargar el primer archivo
mir_embeddings = np.load("../Data_Demo/embedding/binding_doc2vec_mir_embeddings.npy", allow_pickle=True)
print("Shape binding_doc2vec_mir_embeddings:", mir_embeddings.shape)
print("Primeras 2 filas:")
print(mir_embeddings[:2])

# Cargar el segundo archivo
mir_esp_embeddings = np.load("../Data_Demo/embedding/binding_doc2vec_mir_esp_embeddings.npy", allow_pickle=True)
print("\nShape binding_doc2vec_mir_esp_embeddings:", mir_esp_embeddings.shape)
print("Primeras 2 filas:")
print(mir_esp_embeddings[:2])

# Cargar el tercer archivo
lnc_embeddings = np.load("../Data_Demo/embedding/binding_doc2vec_lnc_embeddings.npy", allow_pickle=True)
print("\nShape binding_doc2vec_lnc_embeddings:", lnc_embeddings.shape)
print("Primeras 2 filas:")
print(lnc_embeddings[:2])


Shape binding_doc2vec_mir_embeddings: (45977, 256)
Primeras 2 filas:
[[-9.49265901e-04  1.36499759e-04 -2.94523546e-04 -8.58136103e-04
   1.11119170e-03  3.15690413e-05  9.79572069e-05  6.43619103e-04
   1.83062861e-04  6.66923588e-04  8.88878014e-04  1.87767157e-03
   1.93960709e-03  1.13741634e-03  2.77063344e-04 -1.46066002e-03
  -1.87867763e-03 -4.05033352e-04 -1.28970318e-03  1.57376798e-03
   6.76461495e-05  1.35298655e-03 -8.69611627e-04 -1.54482678e-03
  -9.31896968e-04 -1.13936211e-03  8.78943363e-04  1.82204857e-03
   1.24562765e-03  1.81363709e-03  1.02586346e-03  1.06462953e-03
  -1.54212245e-03  1.89202395e-03 -1.30097813e-03  1.07329874e-03
   5.31993574e-04 -3.35312914e-04  1.08799944e-03  2.69881217e-04
   2.20754882e-04 -1.94763590e-03 -7.88257923e-04  8.08020821e-04
  -5.76827093e-04  7.17100920e-04 -1.59521494e-03  1.37088867e-03
   1.63091440e-03  1.59300980e-03  1.90721010e-03 -1.07112085e-03
  -7.06490828e-04  1.15432986e-03 -5.43512986e-04  1.20199495e-03
  -1.26